<a href="https://colab.research.google.com/github/Murcha1990/ML_AI25/blob/main/Lesson14_PCA%26Embeddings/AI_DimensionReduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тема семинара: отбор признаков

- Фильтрационные методы
- Оберточные методы
- Встроенные методы
- Метод главных компонент или PCA

In [1]:
import pandas as pd

In [49]:
data = pd.read_csv('https://raw.githubusercontent.com/Murcha1990/ML_AI24/refs/heads/main/Lesson14_Features/Pokemon.csv')

In [50]:
data

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,719,Diancie,Rock,Fairy,600,50,100,150,100,150,50,6,True
796,719,DiancieMega Diancie,Rock,Fairy,700,50,160,110,160,110,110,6,True
797,720,HoopaHoopa Confined,Psychic,Ghost,600,80,110,60,150,130,70,6,True
798,720,HoopaHoopa Unbound,Psychic,Dark,680,80,160,60,170,130,80,6,True


Columns description (it's crucial!)


- #: ID for each pokemon
- Name: Name of each pokemon
- Type 1: Each pokemon has a type, this determines weakness/resistance to attacks
- Type 2: Some pokemon are dual type and have 2
- Total: sum of all stats that come after this, a general guide to how strong a pokemon is
- HP: hit points, or health, defines how much damage a pokemon can withstand before fainting
- Attack: the base modifier for normal attacks (eg. Scratch, Punch)
- Defense: the base damage resistance against normal attacks
- SP Atk: special attack, the base modifier for special attacks (e.g. fire blast, bubble beam)
- SP Def: the base damage resistance against special attacks
- Speed: determines which pokemon attacks first each round

In [51]:
# fillna and drop useless cols

display(data.isnull().sum())
data['Type 2'] = data['Type 2'].fillna('No 2nd type')

data.drop(columns=['#', 'Name'], inplace=True)

,0
#,0
Name,0
Type 1,0
Type 2,386
Total,0
HP,0
Attack,0
Defense,0
Sp. Atk,0
Sp. Def,0


In [52]:
X = data.drop(columns='Legendary')
y = data['Legendary'].astype('int')

In [53]:
y.value_counts(normalize=True)

,proportion
Legendary,
0,0.91875
1,0.08125


# Make some default pipeline

In [8]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.8 MB/s eta 0:00:00


In [9]:
from sklearn.feature_selection import SelectKBest, SelectPercentile
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from category_encoders.leave_one_out import LeaveOneOutEncoder
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
import sklearn

In [10]:
# define cat_cols

cat_cols = ['Type 1', 'Type 2']

default_pipeline = Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('scaler_', StandardScaler()),
    ('model_', SVC(kernel='linear'))]
)

In [11]:
from sklearn.model_selection import cross_val_score, cross_validate

In [12]:
# cross_val_score(default_pipeline,
#                         X,
#                         y,
#                         cv=5,
#                         scoring='f1'
#                        ).mean()

In [13]:
cv_res1 = cross_validate(default_pipeline,
                        X,
                        y,
                        cv=5,
                        scoring='f1',
                        n_jobs=-1,
                        return_train_score=True
                       )

In [14]:
cv_res1['test_score'].mean()

np.float64(0.5466128466128466)

# Make pipeline more complicated

In [15]:
# difficult pipeline

pipe_dif = Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('poly_featurizer_', PolynomialFeatures(degree=4)),
    ('scaler_', StandardScaler()),
    ('model_', SVC(kernel='linear'))]
)

In [16]:
cv_res2 = cross_validate(pipe_dif,
                        X,
                        y,
                        cv=5,
                        scoring='f1',
                        n_jobs=-1,
                        return_train_score=True
                       )

cv_res2

{'fit_time': array([0.32344151, 0.36059833, 0.29438162, 0.28287673, 0.16582918]),
 'score_time': array([0.05709672, 0.056638  , 0.07016087, 0.02777267, 0.0557096 ]),
 'test_score': array([0.375     , 0.88888889, 0.5       , 0.66666667, 0.53658537]),
 'train_score': array([0.95145631, 0.89583333, 0.97142857, 0.96153846, 0.98076923])}

In [17]:
cv_res2['train_score'].mean()

np.float64(0.9522051815498418)

In [18]:
cv_res2['test_score'].mean()

np.float64(0.5934281842818427)

train_score - просто класс ! модель получилась сложная, только очевидно переобученная ...

согласны, узнали ?


# Introduce feature selectors

In [19]:
data_tr = pipe_dif[:-1]

In [20]:
data_tr

Pipeline(steps=[('cat_encoder_', LeaveOneOutEncoder(cols=['Type 1', 'Type 2'])),
                ('poly_featurizer_', PolynomialFeatures(degree=4)),
                ('scaler_', StandardScaler())])

In [21]:
X_tr = data_tr.fit_transform(X, y)
print(f'data shape after transformation is {X_tr.shape}')

data shape after transformation is (800, 1001)


1k признаков - многовато, добавим в пайплайн селектор

## Фильтрационные методы

Суть таких методов в том, чтобы для каждого признака посчитать некоторую метрику "связи" с целевым признаком. И в результате оставить топ-K признаков согласно выбранной метрике.

В том числе на лекции обсуждались:

 - статистика хи-квадрат
 - метрика mutual information

In [22]:
from sklearn.feature_selection import f_classif, chi2, mutual_info_classif

In [23]:
# k_best = 30

pipe = Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('poly_featurizer_', PolynomialFeatures(degree=4)),
    ('scaler_', StandardScaler()),
    ('selector_', SelectKBest(score_func=mutual_info_classif, k=100)),
    ('model_', SVC(kernel='linear'))]
)

In [24]:
cv_res = cross_validate(pipe, X, y, cv=5, scoring='f1', return_train_score=True)
cv_res

{'fit_time': array([4.3602047 , 4.72382283, 4.05285811, 4.00717258, 4.84626746]),
 'score_time': array([0.01383281, 0.01704526, 0.01262426, 0.01255488, 0.01834512]),
 'test_score': array([0.26666667, 0.57142857, 0.41666667, 0.43478261, 0.58823529]),
 'train_score': array([0.83168317, 0.65168539, 0.82352941, 0.80412371, 0.87128713])}

In [25]:
# k best нужно подбирать

cv_res['test_score'].mean()

np.float64(0.45555596151504074)

In [26]:
cv_res['train_score'].mean()

np.float64(0.7964617626786084)

## Жадный метод отбора

In [27]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

In [28]:
k_best = 50

rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=k_best, step=30)

In [29]:
X_tr.shape

(800, 1001)

In [30]:
res = rfe.fit_transform(X_tr, y)
display(res.shape)
res

(800, 50)

array([[-0.98555744, -0.44705251, -0.94218651, ..., -0.40535388,
        -0.52135831, -0.72668962],
       [-0.48479877, -0.37458929, -0.94218651, ..., -0.16466941,
        -0.18003271, -0.70651232],
       [ 0.42451538, -0.24576578, -0.94218651, ...,  0.45023305,
         0.73602435, -0.6731154 ],
       ...,
       [-0.16049792,  0.66347512, -0.94218651, ...,  3.97831918,
         2.1076934 ,  4.87819432],
       [-0.16049792,  1.03195307, -0.94218651, ...,  6.06069716,
         2.51608978,  4.87819432],
       [ 1.36562373, -0.13354138, -0.94218651, ...,  1.4680924 ,
         0.19750623,  1.93926564]])

In [31]:
pipe_rfe = Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('poly_featurizer_', PolynomialFeatures(degree=4)),
    ('scaler_', StandardScaler()),
    ('selector_', RFE(LogisticRegression(max_iter=1000),
                      n_features_to_select=20,
                      step=30
                     )),
    ('model_', SVC(kernel='linear'))])

In [32]:
cv_res3 = cross_validate(pipe_rfe, X, y, cv=5, scoring='f1', return_train_score=True)
cv_res3

{'fit_time': array([11.15197206, 17.57898855, 23.6894846 , 11.40862775, 10.03817916]),
 'score_time': array([0.02724457, 0.03820825, 0.01530099, 0.04040241, 0.01527429]),
 'test_score': array([0.23529412, 0.8       , 0.82758621, 0.52173913, 0.64864865]),
 'train_score': array([0.82105263, 0.82978723, 0.82978723, 0.875     , 0.89108911])}

In [33]:
cv_res3['test_score'].mean()

np.float64(0.6066536207254084)

In [34]:
cv_res3['train_score'].mean()

np.float64(0.849343241714989)

## С помощью модели

In [35]:
from sklearn.feature_selection import SelectFromModel

In [36]:
sel = SelectFromModel(LogisticRegression(penalty='l1', max_iter=1000, solver='liblinear'), threshold=0.1)

In [37]:
# пример

res = sel.fit_transform(X_tr, y)
display(res.shape)
res

(800, 35)

array([[-0.7732015 , -0.80122124, -0.31795653, ..., -0.52135831,
        -0.72668962, -0.81966779],
       [-0.33767384, -0.35246512, -0.30715816, ..., -0.18003271,
        -0.70651232, -0.81698899],
       [ 0.6523422 ,  0.66762108, -0.28796106, ...,  0.73602435,
        -0.6731154 , -0.81341726],
       ...,
       [ 0.80566326,  0.55477968,  0.91906366, ...,  2.1076934 ,
         4.87819432,  4.18343684],
       [ 1.06849938,  0.55477968,  0.91906366, ...,  2.51608978,
         4.87819432,  4.18343684],
       [ 2.2512619 ,  1.23182804, -0.18905176, ...,  0.19750623,
         1.93926564,  2.6404483 ]])

In [38]:
pipe_lasso =  Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('poly_featurizer_', PolynomialFeatures(degree=4)),
    ('scaler_', StandardScaler()),
    ('selector_', SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear'),
                                  threshold=1e-5)),
    ('model_', SVC(kernel='linear'))])

In [39]:
cv_res4 = cross_validate(pipe_lasso, X, y, cv=5, scoring='f1', return_train_score=True)
cv_res4

{'fit_time': array([0.15861511, 0.16175961, 0.22881699, 0.20233512, 0.21127605]),
 'score_time': array([0.01555252, 0.01913333, 0.01886225, 0.01794171, 0.01834726]),
 'test_score': array([0.44444444, 0.84615385, 0.68965517, 0.60869565, 0.68421053]),
 'train_score': array([0.92307692, 0.89795918, 0.96078431, 0.93877551, 0.92929293])}

In [40]:
cv_res4['test_score'].mean()

np.float64(0.6546319283003572)

# PCA или метод главных компонент

Цель: создать k новых признаков из какого-либо количества старых признаков, так чтобы
- каждый из новых признаков был линейной комбинацией старых

$z_i = u_1x_{1i} + ... + u_lx{li}$

- и дисперсия $z_i$, то есть новых признаков была максимальной (наиболее информативной)

С точки зрения линеной алгебры, процесс нахождения новых признаков из старых - это процесс проекции старых признаков на некоторую гиперплоскоть (линейное пространство). Как было показано на лекции, базисом этого пространства являются собственные вектора матрицы $X^TX$ - где Х - это центрированная матрица признаков

Тогда чтобы найти новые признаки (главные компоненты) нужно сначала
- найти собственные вектора V матрицы $X^TX$ (вектора должны быть приведены к длине 1)
- произвести матричное умножение Z = XV (то есть сделать проекцию матрицы X на линейное пространство с базисом V)

In [41]:
from sklearn.decomposition import PCA

In [42]:
pca = PCA(n_components=2)

#X = [[1,3],[0,2],[0,0],[3,3]]
X = [[0,1],[-1,0],[-1,-2],[2,1]]
pca.fit_transform(X)

array([[ 0.70710678, -0.70710678],
       [-0.70710678, -0.70710678],
       [-2.12132034,  0.70710678],
       [ 2.12132034,  0.70710678]])

In [43]:
pca.explained_variance_ratio_

array([0.83333333, 0.16666667])

In [44]:
# пример

pca = PCA(n_components = 15)

In [45]:
res = pca.fit_transform(X_tr)
display(res.shape)
res

(800, 15)

array([[-1.98007291e+01,  4.46072632e+00, -4.82046552e-01, ...,
        -2.71943040e-02, -4.35469500e-01, -1.29320941e+00],
       [-1.58785523e+01,  3.10134435e+00, -3.17353060e+00, ...,
         8.35842641e-01, -2.34667207e-01, -1.23773318e+00],
       [-6.24072086e+00, -4.63280109e-01, -9.93003245e+00, ...,
         3.27932890e+00,  4.32694547e-01, -1.25532105e+00],
       ...,
       [ 3.26767745e+01,  1.50340452e+01, -2.46534010e+01, ...,
        -5.00489564e+00,  1.39591872e-02,  3.51950323e+00],
       [ 5.53322261e+01,  1.48563726e+01, -3.86325556e+01, ...,
        -5.42124309e+00,  2.40506303e+00,  4.63244458e-01],
       [ 1.99828053e+01, -4.60646431e+00, -2.56386391e+01, ...,
        -1.78384616e+00,  1.85660560e+00,  1.47080339e+00]])

In [46]:
# суммарная доля объясненной дисперсии исходных признаков

pca.explained_variance_ratio_.sum()

np.float64(0.9438803234251366)

In [47]:
# каждая следующая компонента менее информативна чем предыдущая
pca.explained_variance_ratio_

array([0.45670112, 0.1176569 , 0.08837368, 0.07259347, 0.05161779,
       0.03907593, 0.03147126, 0.02632701, 0.01579202, 0.01454208,
       0.00758009, 0.00692028, 0.00568022, 0.0051811 , 0.00436738])

In [72]:
n_components = 10

pipe_pca = Pipeline([
    ('cat_encoder_', LeaveOneOutEncoder(cols=cat_cols)),
    ('poly_featurizer_', PolynomialFeatures(degree=4)),
    ('scaler_', StandardScaler()),
    ('selector_', PCA(n_components=n_components)),
    ('model_', SVC(kernel='linear'))])

cv_res5 = cross_validate(pipe_pca, X, y, cv=5, scoring='f1', return_train_score=True)
cv_res5

{'fit_time': array([0.89496541, 0.83199644, 0.48769522, 0.59997797, 0.7733736 ]),
 'score_time': array([0.01858544, 0.02964807, 0.02720618, 0.02699876, 0.06213784]),
 'test_score': array([0.44444444, 0.81481481, 0.56      , 0.38095238, 0.61538462]),
 'train_score': array([0.69306931, 0.53658537, 0.70103093, 0.71111111, 0.76767677])}

In [73]:
cv_res5['test_score'].mean()

np.float64(0.5631192511192511)

In [74]:
cv_res5['train_score'].mean()

np.float64(0.6818946958814565)